# Logistic Regression with Class Weighting

## Objective

This project demonstrates how to handle **imbalanced datasets** using Logistic Regression.

We compare:

- Logistic Regression without class weights
- Logistic Regression with `class_weight="balanced"`

The project also shows how to manually compute class weights before training the model.

# Problem Statement

A fintech company wants to detect fraudulent transactions.

The dataset is highly imbalanced:

- Most transactions are legitimate.
- Very few transactions are fraudulent.

The objective is to compare a normal Logistic Regression model with a balanced Logistic Regression model and observe the impact of class weighting.

# What is Class Imbalance?

A dataset is called **imbalanced** when one class contains significantly more samples than another.

Example:

Legitimate Transactions : 90%

Fraud Transactions : 10%

Machine Learning models may become biased toward the majority class.

To solve this problem, class weighting can be used.

# What is Class Weight?

Class Weight assigns a larger penalty to mistakes made on the minority class.

This encourages the model to pay more attention to fraud cases during training.

# Class Weight Formula

\[
\text{Class Weight} =
\frac{\text{Total Samples}}
{\text{Number of Classes} \times \text{Samples in that Class}}
\]

Example:

Total Samples = 12

Classes = 2

Class 0 = 9

Class 1 = 3

Weight(Class 0)

= 12 / (2 × 9)

≈ 0.667

Weight(Class 1)

= 12 / (2 × 3)

= 2.0

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
# Create Dataset

data = {
    "V1": [0.12,-0.33,0.67,-1.02,0.44,-0.28,0.91,-0.19,0.05,-2.75,-3.02,-2.91],
    "V2": [-0.45,0.88,0.21,-0.15,0.09,0.55,-0.62,0.31,-0.88,3.10,2.88,3.45],
    "Amount":[25.50,120.00,45.75,300.10,15.20,89.99,5.00,210.40,60.00,2.50,400.00,1.20],
    "Class":[0,0,0,0,0,0,0,0,0,1,1,1]
}

df = pd.DataFrame(data)

df.head()

,V1,V2,Amount,Class
0,0.12,-0.45,25.50,0
1,-0.33,0.88,120.00,0
2,0.67,0.21,45.75,0
3,-1.02,-0.15,300.10,0
4,0.44,0.09,15.20,0


In [3]:
# Compute Class Weights Manually
def compute_class_weight(
    n_samples,
    n_classes,
    n_samples_in_class
):

    return n_samples / (
        n_classes * n_samples_in_class
    )


total = len(df)

classes = df["Class"].nunique()

weight_0 = compute_class_weight(
    total,
    classes,
    sum(df["Class"] == 0)
)

weight_1 = compute_class_weight(
    total,
    classes,
    sum(df["Class"] == 1)
)

print("Class 0 Weight:", round(weight_0,3))
print("Class 1 Weight:", round(weight_1,3))

Class 0 Weight: 0.667
Class 1 Weight: 2.0


In [4]:
# Define Features and Target
X = df.drop("Class", axis=1)

y = df["Class"]

In [5]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.25,

    stratify=y,

    random_state=42
)

In [6]:
# Feature Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [7]:
# Train Baseline Logistic Regression
baseline = LogisticRegression(

    random_state=42,

    max_iter=1000
)

baseline.fit(
    X_train_scaled,
    y_train
)

LogisticRegression(max_iter=1000, random_state=42)

In [8]:
# Train Balanced Logistic Regression
balanced = LogisticRegression(

    class_weight="balanced",

    random_state=42,

    max_iter=1000
)

balanced.fit(
    X_train_scaled,
    y_train
)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [9]:
# Model Evaluation
def evaluate_model(model,name):

    prediction = model.predict(X_test_scaled)

    print("="*50)

    print(name)

    print("="*50)

    print("Accuracy :",accuracy_score(y_test,prediction))

    print("Precision:",precision_score(y_test,prediction))

    print("Recall   :",recall_score(y_test,prediction))

    print("F1 Score :",f1_score(y_test,prediction))


evaluate_model(
    baseline,
    "Baseline Logistic Regression"
)

evaluate_model(
    balanced,
    "Balanced Logistic Regression"
)

Baseline Logistic Regression
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
Balanced Logistic Regression
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


# Conclusion

The comparison shows how class weighting can improve learning for imbalanced datasets.

Workflow

Dataset

↓

Compute Class Weights

↓

Train-Test Split

↓

Feature Scaling

↓

Baseline Logistic Regression

↓

Balanced Logistic Regression

↓

Performance Comparison